# Part 1 Introduction, Dataset Description, Cleaning & Reshaping

## 1. Project Introduction

Public transit agencies in the U.S. report their annual fuel and energy
consumption to the Federal Transit Administration (FTA) as part of the
**National Transit Database (NTD)**. This project uses the 2023 release of
that data to build a regression model that predicts how much fuel/energy
an agency consumes, based on the type of service, mode of transport, and
other operating details.

This part covers: dataset description, cleaning, and reshaping into a tidy
table. 

**Dataset:** 2023 Fuel and Energy
**Source:** U.S. Department of Transportation / Federal Transit
Administration, via [Data.gov](https://www.transit.dot.gov/ntd/data-product/2023-fuel-and-energy)

## 2. Setup

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 3. Load the raw data

In [3]:
import pandas as pd
path =  r'C:\Users\RaNia PC\Desktop\Axsos\8ehq-7his (1).csv.xlsx'
df = pd.read_excel(path)
df.head()

,agency,city,state,ntd_id,organization_type,reporter_type,report_year,uace_code,uza_name,primary_uza_population,...,compressed_natural_gas_mpg,compressed_natural_gas_mpg_1,hydrogen_mpkg_,hydrogen_mpkg_questionable,other_fuel_mpg,other_fuel_mpg_questionable,electric_propulsion_mi_kwh,electric_propulsion_mi_kwh_1,electric_battery_mi_kwh,electric_battery_mi_kwh_1
0,"King County, dba: King County Metro",Seattle,WA,1,"City, County or Local Government Unit or Depar...",Full Reporter,2023,80389,"Seattle--Tacoma, WA",3544011,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"King County, dba: King County Metro",Seattle,WA,1,"City, County or Local Government Unit or Depar...",Full Reporter,2023,80389,"Seattle--Tacoma, WA",3544011,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"King County, dba: King County Metro",Seattle,WA,1,"City, County or Local Government Unit or Depar...",Full Reporter,2023,80389,"Seattle--Tacoma, WA",3544011,...,NaN,NaN,NaN,NaN,NaN,NaN,0.21,NaN,NaN,NaN
3,"King County, dba: King County Metro",Seattle,WA,1,"City, County or Local Government Unit or Depar...",Full Reporter,2023,80389,"Seattle--Tacoma, WA",3544011,...,NaN,NaN,NaN,NaN,NaN,NaN,0.09,NaN,NaN,NaN
4,"King County, dba: King County Metro",Seattle,WA,1,"City, County or Local Government Unit or Depar...",Full Reporter,2023,80389,"Seattle--Tacoma, WA",3544011,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
df.shape

(1237, 65)

## 4. Explore the raw structure

Each row represents one **transit agency + mode of service** combination for the 2023 reporting year. Columns fall into four groups: agency/admin identifiers, fuel-amount columns (9 fuel types), a second set of plain named fuel-indicator columns, and fuel-efficiency columns (9 fuel types).

**In the README the full column-by-column dictionary.**

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1237 entries, 0 to 1236
Data columns (total 65 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   agency                           1237 non-null   str    
 1   city                             1237 non-null   str    
 2   state                            1237 non-null   str    
 3   ntd_id                           1237 non-null   int64  
 4   organization_type                1237 non-null   str    
 5   reporter_type                    1237 non-null   str    
 6   report_year                      1237 non-null   int64  
 7   uace_code                        1237 non-null   int64  
 8   uza_name                         1237 non-null   str    
 9   primary_uza_population           1237 non-null   int64  
 10  agency_voms                      1237 non-null   int64  
 11  modecd                           1237 non-null   str    
 12  mode_name                      

In [6]:
fuel_columns = [
    "diesel_gal", "gasoline_gal", "liquefied_petroleum_gas_gal",
    "compressed_natural_gas_gal", "bio_diesel_gal", "hydrogen_kg_",
    "other_fuel_gal_gal_equivalent", "electric_propulsion_kwh",
    "electric_battery_kwh",
]
fuel_columns = [c for c in fuel_columns if c in df.columns]
df[fuel_columns].notna().sum().sort_values(ascending=False)

electric_battery_kwh             1237
gasoline_gal                      707
diesel_gal                        558
compressed_natural_gas_gal        209
electric_propulsion_kwh            90
bio_diesel_gal                     73
liquefied_petroleum_gas_gal        54
hydrogen_kg_                        9
other_fuel_gal_gal_equivalent       5
dtype: int64

In [7]:
fully_null_cols = [c for c in df.columns if df[c].isna().sum() == len(df)]
print(f"Fully empty columns: {len(fully_null_cols)}")
fully_null_cols

Fully empty columns: 24


['diesel_gal_questionable',
 'gasoline_gal_questionable',
 'liquefied_petroleum_gas_gal_1',
 'compressed_natural_gas_gal_1',
 'bio_diesel_gal_questionable',
 'hydrogen_kg_questionable',
 'other_fuel_gal_gal_equivalent_1',
 'electric_propulsion_kwh_1',
 'electric_battery_kwh_1',
 'diesel_questionable',
 'gasoline_questionable',
 'liquefied_petroleum_gas_1',
 'compressed_natural_gas_1',
 'hydrogen_questionable',
 'other_fuel_questionable',
 'electric_propulsion_1',
 'electric_battery_questionable',
 'diesel_mpg_questionable',
 'gasoline_mpg_questionable',
 'liquefied_petroleum_gas_mpg_1',
 'hydrogen_mpkg_questionable',
 'other_fuel_mpg_questionable',
 'electric_propulsion_mi_kwh_1',
 'electric_battery_mi_kwh_1']

In [8]:
df.sample(5)

,agency,city,state,ntd_id,organization_type,reporter_type,report_year,uace_code,uza_name,primary_uza_population,...,compressed_natural_gas_mpg,compressed_natural_gas_mpg_1,hydrogen_mpkg_,hydrogen_mpkg_questionable,other_fuel_mpg,other_fuel_mpg_questionable,electric_propulsion_mi_kwh,electric_propulsion_mi_kwh_1,electric_battery_mi_kwh,electric_battery_mi_kwh_1
1188,San Luis Obispo Regional Transit Authority,San Luis Obispo,CA,90206,Independent Public Agency or Authority of Tran...,Full Reporter,2023,79147,"San Luis Obispo, CA",56904,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
156,"City of Nashua, dba: Nashua Transit System",Nashua,NH,10087,"City, County or Local Government Unit or Depar...",Full Reporter,2023,61165,"Nashua, NH--MA",242984,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1156,Los Angeles County Metropolitan Transportation...,Los Angeles,CA,90154,Independent Public Agency or Authority of Tran...,Full Reporter,2023,51445,"Los Angeles--Long Beach--Anaheim, CA",12237376,...,NaN,NaN,NaN,NaN,NaN,NaN,0.1,NaN,NaN,NaN
307,"Maryland Transit Administration, dba: MTA",Baltimore,MD,30034,State Government Unit or Department of Transpo...,Full Reporter,2023,4843,"Baltimore, MD",2212038,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
514,"Brevard Board of County Commissioners, dba: Sp...",Titusville,FL,40063,"City, County or Local Government Unit or Depar...",Full Reporter,2023,67105,"Palm Bay--Melbourne, FL",510675,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 5. Data cleaning

### Step 1 Drop fully empty columns

In [9]:
for c in df.columns:
    if df[c].isna().sum() == len(df):
        df = df.drop(columns=c)

print(f"Shape after dropping fully-empty columns: {df.shape}")
df.info()

Shape after dropping fully-empty columns: (1237, 41)
<class 'pandas.DataFrame'>
RangeIndex: 1237 entries, 0 to 1236
Data columns (total 41 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   agency                         1237 non-null   str    
 1   city                           1237 non-null   str    
 2   state                          1237 non-null   str    
 3   ntd_id                         1237 non-null   int64  
 4   organization_type              1237 non-null   str    
 5   reporter_type                  1237 non-null   str    
 6   report_year                    1237 non-null   int64  
 7   uace_code                      1237 non-null   int64  
 8   uza_name                       1237 non-null   str    
 9   primary_uza_population         1237 non-null   int64  
 10  agency_voms                    1237 non-null   int64  
 11  modecd                         1237 non-null   str    
 12  mode_n

### Step 2  Drop identifier / redundant columns

| Column | Reason for dropping |
|---|---|
| `ntd_id` | Unique per-agency ID; `agency` already identifies the agency |
| `reporter_type` | NTD administrative category, unrelated to fuel/energy use |
| `uza_name` | Redundant with `uace_code` |
| `modecd` | Redundant with `mode_name` |

In [10]:
df = df.drop(columns=[c for c in ["ntd_id", "reporter_type", "uza_name", "modecd"]
                       if c in df.columns])
print(f"Shape after dropping identifier/redundant columns: {df.shape}")
df.info()

Shape after dropping identifier/redundant columns: (1237, 37)
<class 'pandas.DataFrame'>
RangeIndex: 1237 entries, 0 to 1236
Data columns (total 37 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   agency                         1237 non-null   str    
 1   city                           1237 non-null   str    
 2   state                          1237 non-null   str    
 3   organization_type              1237 non-null   str    
 4   report_year                    1237 non-null   int64  
 5   uace_code                      1237 non-null   int64  
 6   primary_uza_population         1237 non-null   int64  
 7   agency_voms                    1237 non-null   int64  
 8   mode_name                      1237 non-null   str    
 9   typeofservicecd                1237 non-null   str    
 10  mode_voms                      1233 non-null   float64
 11  diesel_gal                     558 non-null    float64
 1

## 6. Reshape  fuel amounts and efficiency, kept separate

Fuel-amount and efficiency columns are melted into two independent tidy
tables instead of one combined double melt. This preserves every reported
fuel-amount observation, whether or not that agency also reported an
efficiency value for it.

### Step 3 Melt fuel amount columns, keeping a readable Fuel Type label

In [11]:
FUEL_COLUMNS = [
    "diesel_gal", "gasoline_gal", "liquefied_petroleum_gas_gal",
    "compressed_natural_gas_gal", "bio_diesel_gal", "hydrogen_kg_",
    "other_fuel_gal_gal_equivalent", "electric_propulsion_kwh",
    "electric_battery_kwh",
]
FUEL_COLUMNS = [c for c in FUEL_COLUMNS if c in df.columns]

# Canonical, human-readable fuel type per column — this is what the
# previous version lost by dropping "Type of Fuel" after deriving the unit.
FUEL_LABELS = {
    "diesel_gal": "Diesel",
    "gasoline_gal": "Gasoline",
    "liquefied_petroleum_gas_gal": "LPG",
    "compressed_natural_gas_gal": "CNG",
    "bio_diesel_gal": "Biodiesel",
    "hydrogen_kg_": "Hydrogen",
    "other_fuel_gal_gal_equivalent": "Other Fuel",
    "electric_propulsion_kwh": "Electric Propulsion",
    "electric_battery_kwh": "Electric Battery",
}

id_vars = [c for c in df.columns if c not in FUEL_COLUMNS]

df_fuel_amount = df.melt(
    id_vars=id_vars,
    value_vars=FUEL_COLUMNS,
    var_name="Fuel_Column",
    value_name="Amount Used",
)
df_fuel_amount["Fuel Type"] = df_fuel_amount["Fuel_Column"].map(FUEL_LABELS)
df_fuel_amount["Unit of Fuel"] = np.select(
    [
        df_fuel_amount["Fuel_Column"].str.contains("gal", case=False, na=False),
        df_fuel_amount["Fuel_Column"].str.contains("kg", case=False, na=False),
        df_fuel_amount["Fuel_Column"].str.contains("kwh", case=False, na=False),
    ],
    ["gal", "kg", "kwh"],
    default="",
)
df_fuel_amount = df_fuel_amount.drop(columns="Fuel_Column")
print(f"Shape after melting fuel columns: {df_fuel_amount.shape}")
df_fuel_amount.head()

Shape after melting fuel columns: (11133, 31)


,agency,city,state,organization_type,report_year,uace_code,primary_uza_population,agency_voms,mode_name,typeofservicecd,...,liquefied_petroleum_gas_mpg,compressed_natural_gas_mpg,compressed_natural_gas_mpg_1,hydrogen_mpkg_,other_fuel_mpg,electric_propulsion_mi_kwh,electric_battery_mi_kwh,Amount Used,Fuel Type,Unit of Fuel
0,"King County, dba: King County Metro",Seattle,WA,"City, County or Local Government Unit or Depar...",2023,80389,3544011,2270,Vanpool,DO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Diesel,gal
1,"King County, dba: King County Metro",Seattle,WA,"City, County or Local Government Unit or Depar...",2023,80389,3544011,2270,Demand Response,PT,...,4.97,NaN,NaN,NaN,NaN,NaN,NaN,17880.0,Diesel,gal
2,"King County, dba: King County Metro",Seattle,WA,"City, County or Local Government Unit or Depar...",2023,80389,3544011,2270,Trolleybus,DO,...,NaN,NaN,NaN,NaN,NaN,0.21,NaN,NaN,Diesel,gal
3,"King County, dba: King County Metro",Seattle,WA,"City, County or Local Government Unit or Depar...",2023,80389,3544011,2270,Streetcar Rail,DO,...,NaN,NaN,NaN,NaN,NaN,0.09,NaN,NaN,Diesel,gal
4,"King County, dba: King County Metro",Seattle,WA,"City, County or Local Government Unit or Depar...",2023,80389,3544011,2270,Bus,PT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,43135.0,Diesel,gal


Keep only rows where a fuel amount was actually reported  do not
require efficiency to also be present.

In [12]:
df_fuel_amount = df_fuel_amount.dropna(subset=["Amount Used"])
df_fuel_amount = df_fuel_amount.drop_duplicates()
print(f"Fuel-amount table after dropna (Amount Used only) + drop_duplicates: {df_fuel_amount.shape}")
df_fuel_amount["Fuel Type"].value_counts()

Fuel-amount table after dropna (Amount Used only) + drop_duplicates: (2942, 31)


Fuel Type
Electric Battery       1237
Gasoline                707
Diesel                  558
CNG                     209
Electric Propulsion      90
Biodiesel                73
LPG                      54
Hydrogen                  9
Other Fuel                5
Name: count, dtype: int64

### Step 4 Melt efficiency columns, on their own

In [13]:
EFFICIENCY_COLUMNS = [
    "diesel_mpg", "gasoline_mpg", "liquefied_petroleum_gas_mpg",
    "compressed_natural_gas_mpg", "hydrogen_mpkg_", "other_fuel_mpg",
    "electric_propulsion_mi_kwh", "electric_battery_mi_kwh",
]
EFFICIENCY_COLUMNS = [c for c in EFFICIENCY_COLUMNS if c in df.columns]

# Map each efficiency column to the SAME Fuel Type label used above, so the
# two tables can be joined back together on a shared key. (Biodiesel has no
# matching efficiency column in this dataset — that's expected, not a bug.)
EFFICIENCY_LABELS = {
    "diesel_mpg": "Diesel",
    "gasoline_mpg": "Gasoline",
    "liquefied_petroleum_gas_mpg": "LPG",
    "compressed_natural_gas_mpg": "CNG",
    "hydrogen_mpkg_": "Hydrogen",
    "other_fuel_mpg": "Other Fuel",
    "electric_propulsion_mi_kwh": "Electric Propulsion",
    "electric_battery_mi_kwh": "Electric Battery",
}

eff_id_vars = [c for c in df.columns if c not in EFFICIENCY_COLUMNS]

df_efficiency = df.melt(
    id_vars=eff_id_vars,
    value_vars=EFFICIENCY_COLUMNS,
    var_name="Efficiency_Column",
    value_name="Amount of Efficiency",
)
df_efficiency["Fuel Type"] = df_efficiency["Efficiency_Column"].map(EFFICIENCY_LABELS)
df_efficiency["Efficiency Unit"] = np.select(
    [
        df_efficiency["Efficiency_Column"].str.contains("mpg", case=False, na=False),
        df_efficiency["Efficiency_Column"].str.contains("mpkg", case=False, na=False),
        df_efficiency["Efficiency_Column"].str.contains("kwh", case=False, na=False),
    ],
    ["mpg", "mpkg", "kwh"],
    default="",
)
df_efficiency = df_efficiency.drop(columns="Efficiency_Column")

df_efficiency = df_efficiency.dropna(subset=["Amount of Efficiency"])
df_efficiency = df_efficiency.drop_duplicates()
print(f"Efficiency table after dropna + drop_duplicates: {df_efficiency.shape}")
df_efficiency.head()

Efficiency table after dropna + drop_duplicates: (1656, 32)


,agency,city,state,organization_type,report_year,uace_code,primary_uza_population,agency_voms,mode_name,typeofservicecd,...,liquefied_petroleum_gas,compressed_natural_gas,hydrogen,other_fuel,electric_propulsion,electric_battery,compressed_natural_gas_mpg_1,Amount of Efficiency,Fuel Type,Efficiency Unit
1,"King County, dba: King County Metro",Seattle,WA,"City, County or Local Government Unit or Depar...",2023,80389,3544011,2270,Demand Response,PT,...,1453221,0,0,0,0,NaN,NaN,9.77,Diesel,mpg
4,"King County, dba: King County Metro",Seattle,WA,"City, County or Local Government Unit or Depar...",2023,80389,3544011,2270,Bus,PT,...,0,0,0,0,0,NaN,NaN,6.81,Diesel,mpg
5,"King County, dba: King County Metro",Seattle,WA,"City, County or Local Government Unit or Depar...",2023,80389,3544011,2270,Bus,DO,...,0,0,0,0,0,747864.0,NaN,4.60,Diesel,mpg
6,"King County, dba: King County Metro",Seattle,WA,"City, County or Local Government Unit or Depar...",2023,80389,3544011,2270,Ferryboat,DO,...,0,0,0,0,0,NaN,NaN,0.25,Diesel,mpg
7,Spokane Transit Authority,Spokane,WA,Independent Public Agency or Authority of Tran...,2023,83764,447279,300,Demand Response,DO,...,0,0,0,0,0,NaN,NaN,8.93,Diesel,mpg


### Step 5  Left-join efficiency onto the fuel-amount table

Every fuel-amount row is kept. Efficiency is attached where an agency
reported it for that fuel type, and left as `NaN` where it wasn't —
`NaN` here correctly means "no efficiency reported," not "drop this row."

In [14]:
join_keys = [c for c in eff_id_vars if c in id_vars] + ["Fuel Type"]

df_fuel = df_fuel_amount.merge(
    df_efficiency[join_keys + ["Amount of Efficiency", "Efficiency Unit"]],
    on=join_keys,
    how="left",
)
df_fuel = df_fuel.drop_duplicates()
print(f"Final tidy shape: {df_fuel.shape}")
df_fuel.info()

Final tidy shape: (2942, 33)
<class 'pandas.DataFrame'>
RangeIndex: 2942 entries, 0 to 2941
Data columns (total 33 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   agency                        2942 non-null   str    
 1   city                          2942 non-null   str    
 2   state                         2942 non-null   str    
 3   organization_type             2942 non-null   str    
 4   report_year                   2942 non-null   int64  
 5   uace_code                     2942 non-null   int64  
 6   primary_uza_population        2942 non-null   int64  
 7   agency_voms                   2942 non-null   int64  
 8   mode_name                     2942 non-null   str    
 9   typeofservicecd               2942 non-null   str    
 10  mode_voms                     2935 non-null   float64
 11  diesel                        1495 non-null   float64
 12  gasoline                      2942 non-null 

In [15]:
df_fuel.head()

,agency,city,state,organization_type,report_year,uace_code,primary_uza_population,agency_voms,mode_name,typeofservicecd,...,compressed_natural_gas_mpg_1,hydrogen_mpkg_,other_fuel_mpg,electric_propulsion_mi_kwh,electric_battery_mi_kwh,Amount Used,Fuel Type,Unit of Fuel,Amount of Efficiency,Efficiency Unit
0,"King County, dba: King County Metro",Seattle,WA,"City, County or Local Government Unit or Depar...",2023,80389,3544011,2270,Demand Response,PT,...,NaN,NaN,NaN,NaN,NaN,17880.0,Diesel,gal,9.77,mpg
1,"King County, dba: King County Metro",Seattle,WA,"City, County or Local Government Unit or Depar...",2023,80389,3544011,2270,Bus,PT,...,NaN,NaN,NaN,NaN,NaN,43135.0,Diesel,gal,6.81,mpg
2,"King County, dba: King County Metro",Seattle,WA,"City, County or Local Government Unit or Depar...",2023,80389,3544011,2270,Bus,DO,...,NaN,NaN,NaN,NaN,0.31,7006268.0,Diesel,gal,4.60,mpg
3,"King County, dba: King County Metro",Seattle,WA,"City, County or Local Government Unit or Depar...",2023,80389,3544011,2270,Ferryboat,DO,...,NaN,NaN,NaN,NaN,NaN,5591.0,Diesel,gal,0.25,mpg
4,Spokane Transit Authority,Spokane,WA,Independent Public Agency or Authority of Tran...,2023,83764,447279,300,Demand Response,DO,...,NaN,NaN,NaN,NaN,NaN,30325.0,Diesel,gal,8.93,mpg


In [16]:
print(f"Rows with an efficiency value attached: {df_fuel['Amount of Efficiency'].notna().sum()}")
print(f"Rows with fuel amount but no reported efficiency: {df_fuel['Amount of Efficiency'].isna().sum()}")

Rows with an efficiency value attached: 1608
Rows with fuel amount but no reported efficiency: 1334


## 7. Row/column count through every stage


In [17]:
summary = pd.DataFrame({
    "Stage": [
        "Raw file",
        "After dropping empty & identifier columns",
        "Fuel-amount table (melted, Amount Used only)",
        "Efficiency table (melted, Amount of Efficiency only)",
        "Final tidy table (fuel amounts + left-joined efficiency)",
    ],
    "Rows": [
        1237, 1237,
        df_fuel_amount.shape[0], df_efficiency.shape[0], df_fuel.shape[0],
    ],
    "Columns": [
        65, 37,
        df_fuel_amount.shape[1], df_efficiency.shape[1], df_fuel.shape[1],
    ],
})
summary

,Stage,Rows,Columns
0,Raw file,1237,65
1,After dropping empty & identifier columns,1237,37
2,"Fuel-amount table (melted, Amount Used only)",2942,31
3,"Efficiency table (melted, Amount of Efficiency...",1656,32
4,Final tidy table (fuel amounts + left-joined e...,2942,33


## 8. GGE conversion is required before EDA

`Amount Used` still mixes incompatible units across fuel types (gallons,
kilograms, kWh). This version keeps `Fuel Type` so those units can now be told apart and correctly converted  but the conversion itself hasn't
happened yet. Any totals by fuel type, state, or mode computed directly on
`df_fuel` would still be summing incompatible units.

Before EDA can meaningfully proceed, converts
every fuel type in the **cleaned wide table** (`df_clean.csv`) onto a
common **Gasoline-Gallon-Equivalent (GGE)** basis. Only after that
conversion do fuel-type totals, state comparisons, and mode comparisons
become like-for-like and safe to visualize and summarize.

In [18]:
# Gasoline-Gallon-Equivalent conversion factor per fuel type
GGE_FACTORS = {
    "diesel_gal": 1.12,
    "gasoline_gal": 1.00,
    "liquefied_petroleum_gas_gal": 0.74,
    "compressed_natural_gas_gal": 1.00,
    "bio_diesel_gal": 1.05,
    "hydrogen_kg_": 1.00,
    "other_fuel_gal_gal_equivalent": 1.00,
    "electric_propulsion_kwh": 0.030,
    "electric_battery_kwh": 0.030,
}

##### Convert each fuel column (on the cleaned wide table `df`) to its GGE-equivalent value. Negative reported amounts are treated as missing(not physically meaningful), not converted.



In [19]:
df_gge = df.copy()

for col in FUEL_COLUMNS:
    values = pd.to_numeric(df_gge[col], errors="coerce")
    values = values.mask(values < 0)
    df_gge[f"{col}_GGE"] = values * GGE_FACTORS[col]

GGE_COLUMNS = [f"{c}_GGE" for c in FUEL_COLUMNS]
df_gge["Total_GGE"] = df_gge[GGE_COLUMNS].sum(axis=1, skipna=True)

print(f"GGE dataset: {df_gge.shape[0]} rows, {df_gge.shape[1]} columns")
df_gge[["agency", "state", "mode_name", "Total_GGE"] + GGE_COLUMNS].head()

GGE dataset: 1237 rows, 47 columns


,agency,state,mode_name,Total_GGE,diesel_gal_GGE,gasoline_gal_GGE,liquefied_petroleum_gas_gal_GGE,compressed_natural_gas_gal_GGE,bio_diesel_gal_GGE,hydrogen_kg__GGE,other_fuel_gal_gal_equivalent_GGE,electric_propulsion_kwh_GGE,electric_battery_kwh_GGE
0,"King County, dba: King County Metro",WA,Vanpool,379932.00,NaN,379932.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0
1,"King County, dba: King County Metro",WA,Demand Response,1177253.92,20025.6,940728.0,216500.32,NaN,NaN,NaN,NaN,NaN,0.0
2,"King County, dba: King County Metro",WA,Trolleybus,393399.90,NaN,NaN,NaN,NaN,NaN,NaN,NaN,393399.90,0.0
3,"King County, dba: King County Metro",WA,Streetcar Rail,63673.17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,63673.17,0.0
4,"King County, dba: King County Metro",WA,Bus,175000.20,48311.2,126689.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0


##### Tidy GGE table for EDA  Melt the fuel columns into long format so every fuel type is comparable on the same GGE basis, reusing the same `FUEL_LABELS` mapping from Step 3.

In [20]:
df_gge_fuel = df_gge.melt(
    id_vars=[c for c in df_gge.columns if c not in FUEL_COLUMNS + GGE_COLUMNS],
    value_vars=FUEL_COLUMNS,
    var_name="Fuel_Column",
    value_name="Amount_Original",
)
df_gge_fuel["Fuel Type"] = df_gge_fuel["Fuel_Column"].map(FUEL_LABELS)
df_gge_fuel["GGE_Factor"] = df_gge_fuel["Fuel_Column"].map(GGE_FACTORS)
df_gge_fuel["Amount_Original"] = pd.to_numeric(df_gge_fuel["Amount_Original"], errors="coerce")
df_gge_fuel["GGE"] = df_gge_fuel["Amount_Original"] * df_gge_fuel["GGE_Factor"]

# Keep only reported fuel amounts
df_gge_fuel = df_gge_fuel.loc[df_gge_fuel["Amount_Original"].notna()].copy()

print(f"Tidy GGE EDA table: {df_gge_fuel.shape[0]} rows")
df_gge_fuel[["agency", "state", "mode_name", "Fuel Type", "Amount_Original", "GGE"]].head()

Tidy GGE EDA table: 2942 rows


,agency,state,mode_name,Fuel Type,Amount_Original,GGE
1,"King County, dba: King County Metro",WA,Demand Response,Diesel,17880.0,20025.60
4,"King County, dba: King County Metro",WA,Bus,Diesel,43135.0,48311.20
5,"King County, dba: King County Metro",WA,Bus,Diesel,7006268.0,7847020.16
6,"King County, dba: King County Metro",WA,Ferryboat,Diesel,5591.0,6261.92
7,Spokane Transit Authority,WA,Demand Response,Diesel,30325.0,33964.00


In [21]:
df_gge_fuel.columns

Index(['agency', 'city', 'state', 'organization_type', 'report_year',
       'uace_code', 'primary_uza_population', 'agency_voms', 'mode_name',
       'typeofservicecd', 'mode_voms', 'diesel', 'gasoline',
       'liquefied_petroleum_gas', 'compressed_natural_gas', 'hydrogen',
       'other_fuel', 'electric_propulsion', 'electric_battery', 'diesel_mpg',
       'gasoline_mpg', 'liquefied_petroleum_gas_mpg',
       'compressed_natural_gas_mpg', 'compressed_natural_gas_mpg_1',
       'hydrogen_mpkg_', 'other_fuel_mpg', 'electric_propulsion_mi_kwh',
       'electric_battery_mi_kwh', 'Total_GGE', 'Fuel_Column',
       'Amount_Original', 'Fuel Type', 'GGE_Factor', 'GGE'],
      dtype='str')

In [28]:
df_gge_fuel.to_csv("cleaned_dataset.csv", index=False)

In [29]:
# Windows example
df_gge_fuel.to_csv(r'C:\Users\RaNia PC\Downloads\cleaned_dataset.csv', index=False)

# Mac/Linux example
df_gge_fuel.to_csv('/Users/RaNia PC/Downloads/cleaned_dataset.csv', index=False)